# 🧠 LoRA from scratch : fine-tuning efficace d'un Transformer

**Objectif** : implémenter **LoRA** (*Low-Rank Adaptation*, Hu et al., 2021) **à la main en PyTorch** (sans la librairie `peft`),
l'appliquer à **DistilBERT** sur **SST-2** (critiques de films positives / négatives), et mesurer combien de précision on garde
en n'entraînant qu'une infime partie du modèle.

On compare trois stratégies :
| Stratégie | Ce qu'on entraîne |
|---|---|
| **Head only** | seulement la tête de classification (référence basse) |
| **LoRA, r = 1 … 32** | de petites matrices A et B dans les projections *query* et *value* de l'attention + la tête |
| **Full fine-tuning** | les 66 millions de paramètres (référence haute) |

> 💡 À lancer sur **Google Colab avec un GPU T4** (Exécution → Modifier le type d'exécution). ~20 minutes pour tout le notebook.

## 1. L'idée mathématique
Une couche linéaire calcule `h = W₀ x`. Le fine-tuning la remplace par `W₀ + ΔW`. LoRA suppose que la correction est **de rang faible** :

$$\Delta W = \frac{\alpha}{r} B A, \quad A \in \mathbb{R}^{r \times d_{in}},\; B \in \mathbb{R}^{d_{out} \times r},\; r \ll d \qquad\Rightarrow\qquad h = W_0 x + \frac{\alpha}{r} B (A x)$$

- `W₀` est **gelé**, seuls `A` et `B` sont entraînés : `r × (d_in + d_out)` paramètres au lieu de `d_in × d_out`
  (pour une matrice 768×768 et r = 8 : 12 288 au lieu de 589 824, soit 2 %).
- `B = 0` à l'initialisation : le modèle commence **exactement** comme le modèle pré-entraîné.
- `α / r` garde une amplitude de mise à jour stable quand on change `r`.
- On calcule `B (A x)` (jamais la grande matrice). Après l'entraînement, on peut **fusionner** `W = W₀ + (α/r) B A` : aucune latence ajoutée.

In [ ]:
# CONFIG
MODEL_NAME = "distilbert-base-uncased"
TRAIN_SUBSET = 20_000      # nombre de phrases d'entraînement (67 349 au total)
EPOCHS = 3
BATCH_SIZE = 32
RANKS = [1, 2, 4, 8, 16, 32]
ALPHA = 16
TARGETS = ("q_lin", "v_lin")   # projections query et value de l'attention de DistilBERT
SEED = 0
SMOKE_TEST = False             # True = petit modèle aléatoire + fausses données, pour tester le code en 1 minute sans GPU

In [ ]:
# !pip install -q transformers datasets   # déjà installés sur Colab
import math, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader

torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Appareil :", device)

## 2. La couche LoRA

In [ ]:
class LoRALinear(nn.Module):
    # Couche linéaire gelée + correction de rang faible (alpha / r) * B @ A
    def __init__(self, base: nn.Linear, r=8, alpha=16):
        super().__init__()
        self.base, self.r, self.scaling = base, r, alpha / r
        d_out, d_in = base.weight.shape
        self.lora_A = nn.Parameter(torch.empty(r, d_in, device=base.weight.device))
        self.lora_B = nn.Parameter(torch.zeros(d_out, r, device=base.weight.device))   # B = 0 -> delta W = 0 au départ
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))                           # comme nn.Linear
        for p in self.base.parameters():
            p.requires_grad = False                                                     # W0 gelé
        self.merged = False

    def forward(self, x):
        out = self.base(x)
        if self.merged:
            return out
        return out + self.scaling * (x @ self.lora_A.T @ self.lora_B.T)                # B (A x)

    @torch.no_grad()
    def merge(self):
        if not self.merged:
            self.base.weight += self.scaling * (self.lora_B @ self.lora_A)
            self.merged = True

### Vérifications (ce qui prouve que l'implémentation est correcte)

In [ ]:
torch.manual_seed(0)
layer = LoRALinear(nn.Linear(768, 768), r=8, alpha=16)
x = torch.randn(5, 768)

# 1) au départ, la couche LoRA est identique à la couche d'origine (B = 0)
assert torch.allclose(layer(x), layer.base(x))
# 2) nombre de paramètres entraînables = r * (d_in + d_out)
n_train = sum(p.numel() for p in layer.parameters() if p.requires_grad)
assert n_train == 8 * (768 + 768), n_train
# 3) après entraînement (on simule B != 0), delta W est exactement de rang r
with torch.no_grad():
    layer.lora_B.normal_()
assert torch.linalg.matrix_rank(layer.scaling * layer.lora_B @ layer.lora_A) == 8
# 4) la fusion donne exactement les mêmes sorties
y_before = layer(x)
layer.merge()
assert torch.allclose(layer(x), y_before, atol=1e-4)
print(f"OK : {n_train:,} paramètres entraînables au lieu de {768*768:,} ({100*n_train/(768*768):.1f} %)")

## 3. Injecter LoRA dans DistilBERT
On remplace chaque couche nommée `q_lin` / `v_lin` (6 blocs × 2 = 12 matrices) par une `LoRALinear`, puis on gèle tout
sauf les matrices LoRA et la tête de classification (`pre_classifier`, `classifier`), créée aléatoirement.

In [ ]:
HEAD = ("pre_classifier", "classifier")

def inject_lora(model, targets, r, alpha):
    replaced = 0
    for _, parent in list(model.named_modules()):
        for child_name, child in list(parent.named_children()):
            if child_name in targets and isinstance(child, nn.Linear):
                setattr(parent, child_name, LoRALinear(child, r=r, alpha=alpha))
                replaced += 1
    return replaced

def freeze_all_but(model, keep):
    for name, p in model.named_parameters():
        p.requires_grad = any(k in name for k in keep)

def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return trainable, total

def load_model():
    from transformers import AutoModelForSequenceClassification, DistilBertConfig, DistilBertForSequenceClassification
    if SMOKE_TEST:
        cfg = DistilBertConfig(vocab_size=1000, dim=64, hidden_dim=128, n_layers=2, n_heads=4, num_labels=2)
        return DistilBertForSequenceClassification(cfg)
    return AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

## 4. Les données : SST-2

In [ ]:
def get_loaders():
    if SMOKE_TEST:   # fausses données : la classe 0 utilise la moitié basse du vocabulaire, la classe 1 la moitié haute
        g = torch.Generator().manual_seed(0)
        labels = torch.randint(0, 2, (256,), generator=g)
        ids = torch.randint(5, 500, (256, 16), generator=g) + labels[:, None] * 495
        rows = [{"input_ids": i, "attention_mask": torch.ones_like(i), "labels": l} for i, l in zip(ids, labels)]
        return DataLoader(rows[:200], batch_size=32, shuffle=True), DataLoader(rows[200:], batch_size=64)
    from datasets import load_dataset
    from transformers import AutoTokenizer, DataCollatorWithPadding
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    ds = load_dataset("nyu-mll/glue", "sst2")
    ds["train"] = ds["train"].shuffle(seed=0).select(range(TRAIN_SUBSET))
    ds = ds.map(lambda b: tok(b["sentence"], truncation=True, max_length=128), batched=True)
    ds = ds.rename_column("label", "labels").remove_columns(["sentence", "idx"])
    collate = DataCollatorWithPadding(tok)
    return (DataLoader(ds["train"], batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate),
            DataLoader(ds["validation"], batch_size=128, collate_fn=collate))

train_dl, val_dl = get_loaders()
print("Lots d'entraînement :", len(train_dl), "| exemples de validation :", len(val_dl.dataset))

## 5. Entraînement et évaluation
AdamW, taux d'apprentissage décroissant linéairement. LR plus grand pour LoRA et la tête (peu de paramètres partant de zéro)
que pour le fine-tuning complet (un grand pas abîmerait le modèle pré-entraîné).

In [ ]:
@torch.no_grad()
def accuracy(model):
    model.eval()
    correct = total = 0
    for batch in val_dl:
        batch = {k: v.to(device) for k, v in batch.items()}
        pred = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).logits.argmax(-1)
        correct += (pred == batch["labels"]).sum().item()
        total += len(pred)
    return correct / total

def run(mode, r=None, lr=None, epochs=EPOCHS):
    torch.manual_seed(SEED)
    model = load_model()
    if mode == "lora":
        inject_lora(model, TARGETS, r, ALPHA)
        freeze_all_but(model, ("lora_A", "lora_B") + HEAD)
    elif mode == "head":
        freeze_all_but(model, HEAD)
    model.to(device)
    trainable, total = count_params(model)
    lr = lr or (2e-5 if mode == "full" else 5e-4)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=0.01)
    steps = epochs * len(train_dl)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: max(0.0, 1 - s / steps))
    t0 = time.time()
    for epoch in range(epochs):
        model.train()
        for batch in train_dl:
            batch = {k: v.to(device) for k, v in batch.items()}
            model(**batch).loss.backward()
            opt.step(); sched.step(); opt.zero_grad()
        acc = accuracy(model)
        print(f"  [{mode}{'' if r is None else f' r={r}'}] époque {epoch + 1} : précision validation = {acc:.4f}")
    if mode == "lora":   # la fusion ne doit rien changer aux prédictions
        before = acc
        for m in model.modules():
            if isinstance(m, LoRALinear):
                m.merge()
        assert abs(accuracy(model) - before) < 1e-6
    name = {"head": "Head only", "full": "Full fine-tuning"}.get(mode, f"LoRA r={r}")
    return {"méthode": name, "mode": mode, "r": r, "param. entraînables": trainable,
            "% du modèle": round(100 * trainable / total, 2), "précision": round(acc, 4),
            "temps (s)": round(time.time() - t0, 1)}

In [ ]:
results = [run("head")]
for r in RANKS:
    results.append(run("lora", r=r))
results.append(run("full"))
table = pd.DataFrame(results).drop(columns=["mode", "r"])
table

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.2))
lora = [x for x in results if x["mode"] == "lora"]
ax.plot([x["param. entraînables"] for x in lora], [x["précision"] for x in lora], "o-", label="LoRA (r = 1..32)")
for x in lora:
    ax.annotate(f"r={x['r']}", (x["param. entraînables"], x["précision"]), textcoords="offset points", xytext=(4, -11), fontsize=8)
for mode, marker, label in [("head", "s", "Head only"), ("full", "^", "Full fine-tuning")]:
    pts = [x for x in results if x["mode"] == mode]
    ax.scatter([p["param. entraînables"] for p in pts], [p["précision"] for p in pts], marker=marker, s=70, label=label, zorder=3)
ax.set_xscale("log"); ax.set_xlabel("Paramètres entraînables (échelle log)"); ax.set_ylabel("Précision validation SST-2")
ax.grid(alpha=0.3); ax.legend(); plt.tight_layout(); plt.show()

## 6. Résultats obtenus (Colab, GPU T4, une graine, 20 000 phrases, 3 époques)

| Méthode | Param. entraînables | % du modèle | Précision val. | Temps |
|---|---|---|---|---|
| Head only | 592 130 | 0,88 % | 0,8326 | 66 s |
| LoRA r=1 | 610 562 | 0,91 % | 0,8796 | 140 s |
| LoRA r=2 | 628 994 | 0,94 % | **0,8922** | 143 s |
| LoRA r=4 | 665 858 | 0,99 % | 0,8750 | 142 s |
| LoRA r=8 | 739 586 | 1,10 % | 0,8842 | 142 s |
| LoRA r=16 | 887 042 | 1,32 % | 0,8807 | 143 s |
| LoRA r=32 | 1 181 954 | 1,75 % | 0,8911 | 144 s |
| Full fine-tuning | 66 955 010 | 100 % | **0,9060** | 225 s |

**Ce qu'on en retient**
- **LoRA récupère ~80 % de l'écart** entre la tête seule (83,3 %) et le fine-tuning complet (90,6 %).
  Les matrices LoRA n'ajoutent que `12 × 1 536 × r = 18 432 r` paramètres (36 864 pour r = 2, soit **0,06 %** du modèle) ;
  la plupart des paramètres entraînables viennent de la tête de classification.
- **Le rang ne change presque rien** : toutes les valeurs LoRA sont entre 87,5 % et 89,2 %. Avec 872 phrases de validation,
  l'erreur standard d'une précision de ~88 % est `√(0,88 × 0,12 / 872) ≈ ±1,1 point` : les écarts entre rangs sont **du bruit**
  (une seule graine). Un très petit rang suffit, comme dans l'article original.
- LoRA est ~37 % plus rapide ici ; son vrai gain est la **mémoire** (pas d'états d'optimiseur pour les 66 M poids gelés)
  et le **stockage** (quelques Mo par tâche au lieu d'une copie complète du modèle).

## Références
- Hu et al., *LoRA: Low-Rank Adaptation of Large Language Models*, ICLR 2022.
- Sanh et al., *DistilBERT, a distilled version of BERT*, 2019.